# Setup

To run this notebook on Colab, a few setup steps are required. Follow along step by step:

1. **Clone the `dlfb` library**  
   First, clone the repository that contains the `dlfb` library.

In [ ]:
%cd /content
!rm -rf ./dlfb-pytorch-clone/
!git clone "https://github.com/deep-learning-for-biology/deep-learning-for-biology-pytorch.git" dlfb-pytorch-clone --branch main
%cd dlfb-pytorch-clone

2. **Install dependencies**  
   Once the library is cloned, install the required dependencies.

In [ ]:
%%bash
pip install -e '.[cancer]'

3. **Providion the datasets**  
   You’ll then need to access and download the necessary datasets for this chapter.

In [ ]:
from google.colab import auth

auth.authenticate_user()
# NOTE: exclude models with '--no-models' flag
!dlfb-provision --chapter cancer

4. **Load the `dlfb` package**  
   Finally, load the `dlfb` package.  
   - ⚠️ Note: Loading can sometimes be finicky. If you encounter issues, simply **restart the runtime**. All previously downloaded data and installed packages will persist, so you can re-run the load step without repeating everything.

In [ ]:
try:
  import dlfb_pytorch
except ImportError as exc:
  import site
  site.main()
  import dlfb_pytorch

import inspect
import random

import numpy as np
import torch
from dlfb_pytorch.utils.display import display

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 5. Detecting Skin Cancer in Medical Images


## 5.1. Biology Primer
### 5.1.1. Skin Cancer
### 5.1.2. Causes and Risk Factors
### 5.1.3. How Skin Cancer Is Diagnosed
### 5.1.4. Image-Based Skin Cancer Detection


## 5.2. Machine Learning Primer
### 5.2.1. Convolutional Neural Networks
### 5.2.2. Understanding a Convolution
### 5.2.3. Understanding Dimensions
### 5.2.4. Pooling
### 5.2.5. Other Components of a CNN
### 5.2.6. ResNets


## 5.3. Exploring the Data
### 5.3.1. A First Glimpse


In [ ]:
import re
from pathlib import Path

from dlfb_pytorch.utils.context import assets

image_file = next(Path(assets("cancer/datasets/raw")).rglob("*.jpg"))

print(rf"One of the images: {re.sub('^.*?datasets/', '', str(image_file))}")

In [ ]:
import pandas as pd


def load_metadata(data_dir: str) -> pd.DataFrame:
  metadata = []
  for path in Path(data_dir).rglob("*.jpg"):
    split, class_name, _ = path.parts[-3:]
    metadata.append(
      {
        "split_orig": split,
        "class_orig": class_name,
        "full_path": str(path),
      }
    )
  return pd.DataFrame(metadata).rename_axis("frame_id").reset_index()


metadata = load_metadata(assets("cancer/datasets/raw"))
print(metadata)

In [ ]:
counts = pd.crosstab(
  metadata["class_orig"], metadata["split_orig"], margins=True
)
print(counts)

In [ ]:
fig = counts.drop(["All"], axis=1).drop(["All"], axis=0).plot.barh(stacked=True)
fig.set_xlabel("Number of images")
fig.set_ylabel("Skin lesion type")
fig.set_title("Class distribution");

### 5.3.2. Previewing the Images


In [ ]:
from PIL import Image

Image.open(metadata["full_path"].iloc[0])

In [ ]:
import matplotlib.pyplot as plt


def show_random_image(metadata: pd.DataFrame, class_name: str) -> plt.Figure:
  record = (
    metadata[metadata["class_orig"] == class_name]
    .sample(1)
    .to_dict(orient="records")[0]
  )
  fig = plt.figure(figsize=(4, 4))
  plt.imshow(Image.open(record["full_path"]))
  plt.title(record["class_orig"].capitalize())
  return fig

In [ ]:
show_random_image(metadata, "melanoma");

In [ ]:
def plot_random_image_grid(
  metadata: pd.DataFrame, ncols: int = 3
) -> plt.Figure:
  """Display a random example image from each class in a grid."""
  records = metadata.groupby("class_orig").sample(1).to_dict(orient="records")
  nrows = (len(records) + ncols - 1) // ncols
  fig, axes = plt.subplots(nrows, ncols, figsize=(10, 2.5 * nrows))
  axes = axes.flatten()

  for record, ax in zip(records, axes):
    ax.imshow(Image.open(record["full_path"]))
    ax.set_title(record["class_orig"].capitalize())
    ax.axis("off")

  plt.tight_layout()
  return fig

In [ ]:
plot_random_image_grid(metadata);

### 5.3.3. Addressing Dataset Issues


In [ ]:
import numpy as np

np.random.seed(seed=42)
splits = {"train": 0.7, "valid": 0.20, "test": 0.10}

metadata["split"] = np.random.choice(
  list(splits.keys()), p=list(splits.values()), size=metadata.shape[0]
)

counts = pd.crosstab(metadata["class_orig"], metadata["split"], margins=True)
print(counts)

In [ ]:
counts.drop(["All"], axis=1).drop(["All"], axis=0).plot.barh(stacked=True);

#### 5.3.3.1. Balancing the Batches
#### 5.3.3.2. Augmenting the Dataset


In [ ]:
from dlfb_pytorch.cancer.train.handlers.augmentors import rich_augmentor

display([rich_augmentor])

In [ ]:
image = Image.open(
  metadata[metadata["class_orig"] == "melanoma"]
  .sample(1)["full_path"]
  .values[0]
)

_, axes = plt.subplots(3, 3, figsize=(9, 9))
axes = axes.flatten()
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
image_np = np.asarray(image, dtype=np.float32) / 255.0
for ax in axes[1:]:
  ax.imshow(np.clip(rich_augmentor(image_np), 0, 1))
  ax.set_title("Augmented")
  ax.axis("off")
plt.tight_layout()

#### 5.3.3.3. Preprocessing the Images


In [ ]:
from dlfb_pytorch.cancer.dataset.preprocessors import (
  center_crop,
  resize_preserve_aspect,
)

display([resize_preserve_aspect, center_crop])

In [ ]:
from dlfb_pytorch.cancer.dataset.preprocessors import rescale_image

display([rescale_image])

In [ ]:
original_image = Image.open(metadata.iloc[0]["full_path"])
image = np.array(original_image)
image = resize_preserve_aspect(image)
image = center_crop(image)
image = rescale_image(image)

_, axes = plt.subplots(1, 2, figsize=(6, 3))
axes = axes.flatten()
axes[0].imshow(original_image)
axes[0].set_title("Original")
axes[1].imshow(image)
axes[1].set_title("Preprocessed");

#### 5.3.3.4. Data Storage with Memory-Mapped Arrays


In [ ]:
from tempfile import TemporaryFile

images_on_disk = np.memmap(
  TemporaryFile(), dtype="float32", mode="w+", shape=(10, 224, 224, 3)
)
for i, full_path in enumerate(metadata[:10]["full_path"]):
  image = Image.open(full_path)
  image = np.array(image)
  image = resize_preserve_aspect(image)
  image = center_crop(image)
  image = rescale_image(image)
  images_on_disk[i, :, :, :] = image
images_on_disk.flush()

### 5.3.4. Building a DatasetBuilder


In [ ]:
from dlfb_pytorch.cancer.dataset.builder import DatasetBuilder

display([DatasetBuilder])

#### 5.3.4.1. Loading the Metadata and Images


In [ ]:
from dlfb_pytorch.cancer.dataset import Dataset

display([Dataset])

In [ ]:
from dlfb_pytorch.cancer.dataset import Images

display([Images])

#### 5.3.4.2. Batching the Data


In [ ]:
from dlfb_pytorch.cancer.train.handlers import BatchHandler

display([BatchHandler])

### 5.3.5. Readying the Dataset


In [ ]:
from dlfb_pytorch.cancer.train.handlers.augmentors import rich_augmentor
from dlfb_pytorch.cancer.train.handlers.samplers import balanced_sampler

np.random.seed(seed)

builder = DatasetBuilder(data_dir=assets("cancer/datasets/raw"))

dataset_splits = builder.build(
  splits={"train": 0.7, "valid": 0.20, "test": 0.10},
  image_size=(224, 224, 3),
)

print(dataset_splits["train"].metadata)

In [ ]:
train_batcher = BatchHandler(sampler=balanced_sampler, augmentor=rich_augmentor)
train_batches = train_batcher.get_batches(
  dataset_splits["train"], batch_size=32,
)
batch = next(train_batches)

In [ ]:
batch["images"].shape

In [ ]:
plt.imshow(batch["images"][0]);

## 5.4. Building Skin Cancer Classification Models
### 5.4.1. Loading the Flax ResNet50 Model


In [ ]:
import requests

# Load the example image from the documentation.
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
plt.imshow(image);

In [ ]:
from transformers import AutoImageProcessor, ResNetForImageClassification

resnet_model = ResNetForImageClassification.from_pretrained(
  "microsoft/resnet-50"
)
image_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")

In [ ]:
inputs = image_processor(images=image, return_tensors="pt")
inputs.keys()

In [ ]:
plt.imshow(inputs["pixel_values"].permute(0, 2, 3, 1)[0]);

In [ ]:
outputs = resnet_model(**inputs)
logits = outputs.logits
predicted_class_idx = logits.argmax(dim=-1)
print(
  "Predicted class:", resnet_model.config.id2label[predicted_class_idx.item()]
)

### 5.4.2. Extracting the ResNet Backbone


In [ ]:
backbone = resnet_model.resnet

In [ ]:
with torch.no_grad():
  outputs = resnet_model.resnet(inputs["pixel_values"])
last_hidden_state = outputs.last_hidden_state
last_hidden_state.shape

In [ ]:
def display_feature_maps(feature_map, ncols=8):
  """Plot grid of the first 64 feature maps."""
  num_features = feature_map.shape[0]
  nrows = (num_features + ncols - 1) // ncols
  _, axes = plt.subplots(nrows, ncols, figsize=(ncols, nrows))
  axes = axes.flatten()

  for i, (ax, feature) in enumerate(zip(axes, feature_map)):
    ax.imshow(feature, cmap="viridis")
    ax.axis("off")

  plt.tight_layout()
  plt.show()


display_feature_maps(last_hidden_state[0, 0:64, ...])

In [ ]:
_, ax = plt.subplots(1, 1, figsize=(6, 2))
ax.plot(np.squeeze(outputs.pooler_output), c="grey")
plt.title(f"Shape of outputs after pooling: {outputs.pooler_output.shape}");

### 5.4.3. Building the SkinLesionClassifierHead


In [ ]:
from dlfb_pytorch.cancer.model import SkinLesionClassifierHead

display([SkinLesionClassifierHead])

### 5.4.4. Building Our Models
#### 5.4.4.1. SimpleCNN Model as a Baseline


In [ ]:
from dlfb_pytorch.cancer.model.cnn import SimpleCnn

display([SimpleCnn])

In [ ]:
from dlfb_pytorch.cancer.model.cnn import CnnBackbone

display([CnnBackbone])

In [ ]:
# In PyTorch, there is no need for a custom TrainState class.
# The model and optimizer objects manage their own state directly.

#### 5.4.4.2. Building the ResNetFromScratch Model


In [ ]:
from dlfb_pytorch.cancer.model.resnet import ResNetFromScratch

display([ResNetFromScratch])

# 6. Dictionary of pretrained ResNet models from Hugging Face.
#### 6.0.0.1. Building the FinetunedResNet Model


In [ ]:
from dlfb_pytorch.cancer.model.resnet import FinetunedResNet

display([FinetunedResNet])

#### 6.0.0.2. Freezing the Backbone with FinetunedHeadResNet


In [ ]:
from dlfb_pytorch.cancer.model.resnet import FinetunedHeadResNet

display([FinetunedHeadResNet])

#### 6.0.0.3. More Customization Options


In [ ]:
from dlfb_pytorch.cancer.model.resnet import PartiallyFinetunedResNet

display([PartiallyFinetunedResNet])

## 6.1. Training the Models
### 6.1.1. The Training Loop


In [ ]:
from dlfb_pytorch.cancer.train import train

display([train])

#### 6.1.1.1. The Training Step


In [ ]:
from dlfb_pytorch.cancer.train import train_step

display([train_step])

#### 6.1.1.2. The Evaluation Step


In [ ]:
from dlfb_pytorch.cancer.train import eval_step

display([eval_step])

#### 6.1.1.3. Evaluation Metrics


In [ ]:
from dlfb_pytorch.cancer.train import compute_metrics

display([compute_metrics])

#### 6.1.1.4. Faster Evaluation Metrics
### 6.1.2. Creating the Multiclass Dataset


In [ ]:
from dlfb_pytorch.cancer.dataset.preprocessors import skew, crop, resnet

np.random.seed(seed)
torch.manual_seed(seed)

dataset_splits = DatasetBuilder(
  data_dir=assets("cancer/datasets/raw"),
).build(
  preprocessors=[skew, crop, resnet],
  image_size=(224, 224, 3),
  splits={"train": 0.70, "valid": 0.20, "test": 0.10},
)

num_classes = dataset_splits["train"].num_classes

### 6.1.3. Training the Baseline Model


In [ ]:
from dlfb_pytorch.cancer.train.handlers.samplers import repeating_sampler

learning_rate = 0.001
num_steps = 2000

model = SimpleCnn(num_classes=num_classes).to(device)
optimizer = model.create_optimizer(lr=learning_rate, weight_decay=0.0)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_steps=num_steps,
  batch_size=32,
  preprocessor=crop,
  sampler=repeating_sampler,
  augmentor=None,
  eval_every=100,
  store_path=assets("cancer/models/simple_cnn"),
)

In [ ]:
from dlfb_pytorch.cancer.inspect import plot_learning

plot_learning(metrics);

In [ ]:
from dlfb_pytorch.cancer.inspect import plot_classified_images
from dlfb_pytorch.cancer.train import get_predictions

predictions = get_predictions(model, dataset_splits["valid"], crop, 32)
plot_classified_images(
  predictions, dataset_splits["valid"], crop, max_images=8
);

### 6.1.4. Training the ResNetFromScratch Model


In [ ]:
model = ResNetFromScratch(num_classes=num_classes).to(device)
optimizer = model.create_optimizer(lr=learning_rate, weight_decay=0.0)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_steps=num_steps,
  batch_size=32,
  preprocessor=crop,
  sampler=repeating_sampler,
  augmentor=None,
  eval_every=100,
  store_path=assets("cancer/models/resnet_from_scratch"),
)

In [ ]:
plot_learning(metrics);

In [ ]:
predictions = get_predictions(state, dataset_splits["valid"], resnet, 32)
plot_classified_images(
  predictions, dataset_splits["valid"], crop, max_images=8
);

### 6.1.5. Training the FinetunedHeadResNet Model


In [ ]:
from dlfb_pytorch.cancer.dataset.preprocessors import resnet

display(
  [
    'IMAGE_PROCESSOR = AutoImageProcessor.from_pretrained("microsoft/resnet-50")',
    resnet
  ]
)

In [ ]:
model = FinetunedHeadResNet(num_classes=num_classes).to(device)
optimizer = model.create_optimizer(lr=learning_rate, weight_decay=0.0)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_steps=num_steps,
  batch_size=32,
  preprocessor=resnet,
  sampler=repeating_sampler,
  augmentor=None,
  eval_every=100,
  store_path=assets("cancer/models/resnet_just_head"),
)

In [ ]:
plot_learning(metrics);

### 6.1.6. Training the `FinetunedResNet` Model


In [ ]:
model = FinetunedResNet(num_classes=num_classes).to(device)
optimizer = model.create_optimizer(lr=learning_rate, weight_decay=0.0)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_steps=num_steps,
  batch_size=32,
  preprocessor=resnet,
  sampler=repeating_sampler,
  augmentor=None,
  eval_every=100,
  store_path=assets("cancer/models/resnet50_basic"),
)

In [ ]:
plot_learning(metrics);

### 6.1.7. Optimizing the `FinetunedResNet` Model
#### 6.1.7.1. Learning Rate Schedule


In [ ]:
import math

learning_rate = 0.001
num_steps = 2000
warmup_steps = int(num_steps * 0.2)
init_value = 0.0001
end_value = 0.00001

def warmup_cosine_schedule(step):
  if step < warmup_steps:
    return init_value / learning_rate + (1 - init_value / learning_rate) * step / warmup_steps
  progress = (step - warmup_steps) / (num_steps - warmup_steps)
  return (end_value / learning_rate) + 0.5 * (1 - end_value / learning_rate) * (1 + math.cos(math.pi * progress))

lrs = [learning_rate * warmup_cosine_schedule(i) for i in range(num_steps)]

plt.scatter(range(num_steps), lrs)
plt.title("Learning Rate over Steps")
plt.ylabel("Learning Rate")
plt.xlabel("Step");

#### 6.1.7.2. Data Augmentation
#### 6.1.7.3. Regularization via Dropout and `adamw`
### 6.1.8. Training the Optimized `FinetunedResNet` Model


In [ ]:
from torch.optim.lr_scheduler import LambdaLR

model = FinetunedResNet(
  num_classes=num_classes, dropout_rate=0.7
).to(device)
optimizer = model.create_optimizer(lr=learning_rate, weight_decay=1e-4)
scheduler = LambdaLR(optimizer, lr_lambda=warmup_cosine_schedule)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_steps=num_steps,
  batch_size=32,
  preprocessor=resnet,
  sampler=repeating_sampler,
  augmentor=rich_augmentor,
  eval_every=100,
  scheduler=scheduler,
  store_path=assets("cancer/models/resnet50_optimized"),
)

In [ ]:
plot_learning(metrics);

In [ ]:
predictions = get_predictions(state, dataset_splits["valid"], resnet, 32)
plot_classified_images(
  predictions, dataset_splits["valid"], crop, max_images=8
);

### 6.1.9. Further Improving the Model


## 6.2. Summary
